In [2]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import torch.nn.functional as F
from tqdm import tqdm
import os
import sys
import time
from torch.utils.data import Dataset, DataLoader
import cv2
import random
import glob
import numpy as np
import torchvision.transforms.functional as TF
import einops

seed=42
random.seed(seed)
torch.manual_seed(seed)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"{device} is avaialable!")

cuda is avaialable!


In [ ]:
#----------------------------------------------------------------------------------#
#                    M O D E L         A R C H I T E C T U R E                     #
#----------------------------------------------------------------------------------#


class IlluminationEstimator(nn.Module):
    def __init__(self, n_fea_middle, n_fea_in=4, n_fea_out=3):
        super(IlluminationEstimator, self).__init__()
        self.conv1 = nn.Conv2d(n_fea_in, n_fea_middle, kernel_size=1, bias=True)
        self.depth_conv = nn.Conv2d(
            n_fea_middle, n_fea_middle, kernel_size=5, padding=2, bias=True, groups=n_fea_in)
        self.conv2 = nn.Conv2d(n_fea_middle, n_fea_out, kernel_size=1, bias=True)

    def forward(self, img):
        mean_c = img.mean(dim=1).unsqueeze(1)
        input_tensor = torch.cat([img, mean_c], dim=1)

        x_1 = self.conv1(input_tensor)
        illu_fea = self.depth_conv(x_1)
        illu_map = self.conv2(illu_fea)
        return illu_fea, illu_map

class IG_MSA(nn.Module):
    def __init__(self, dim, dim_head=64, heads=32):
        super().__init__()
        self.num_heads = heads
        self.dim_head = dim_head
        self.to_q = nn.Linear(dim, dim_head * heads, bias=False)
        self.to_k = nn.Linear(dim, dim_head * heads, bias=False)
        self.to_v = nn.Linear(dim, dim_head * heads, bias=False)
        self.rescale = nn.Parameter(torch.ones(heads, 1, 1))
        self.proj = nn.Linear(dim_head * heads, dim, bias=True)
        self.pos_emb = nn.Sequential(
            nn.Conv2d(dim, dim, 3, 1, 1, bias=False, groups=dim),
            nn.GELU(),
            nn.Conv2d(dim, dim, 3, 1, 1, bias=False, groups=dim),
        )
        self.dim = dim

    def forward(self, x_in, illu_fea_trans):
        b, h, w, c = x_in.shape
        x = x_in.reshape(b, h * w, c)

        q_inp = self.to_q(x)
        k_inp = self.to_k(x)
        v_inp = self.to_v(x)

        illu_attn = illu_fea_trans

        q, k, v, illu_attn = map(
            lambda t: einops.rearrange(t, 'b n (h d) -> b h n d', h=self.num_heads),
            (q_inp, k_inp, v_inp, illu_attn.flatten(1, 2))
        )

        v = v * illu_attn

        q = q.transpose(-2, -1)
        k = k.transpose(-2, -1)
        v = v.transpose(-2, -1)

        q = F.normalize(q, dim=-1, p=2)
        k = F.normalize(k, dim=-1, p=2)

        attn = (k @ q.transpose(-2, -1)) * self.rescale
        attn = attn.softmax(dim=-1)

        x = attn @ v
        x = x.permute(0, 3, 1, 2)
        x = x.reshape(b, h * w, self.num_heads * self.dim_head)

        out_c = self.proj(x).view(b, h, w, c)
        out_p = self.pos_emb(v_inp.reshape(b, h, w, c).permute(0, 3, 1, 2)).permute(0, 2, 3, 1)

        return out_c + out_p

class PreNorm(nn.Module):
    def __init__(self, dim, fn):
        super().__init__()
        self.fn = fn
        self.norm = nn.LayerNorm(dim)

    def forward(self, x, *args, **kwargs):
        x = self.norm(x)
        return self.fn(x, *args, **kwargs)

class FeedForward(nn.Module):
    def __init__(self, dim, mult=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(dim, dim * mult, 1, 1, bias=False),
            nn.GELU(),
            nn.Conv2d(dim * mult, dim * mult, 3, 1, 1, bias=False, groups=dim * mult),
            nn.GELU(),
            nn.Conv2d(dim * mult, dim, 1, 1, bias=False),
        )

    def forward(self, x):
        out = self.net(x.permute(0, 3, 1, 2).contiguous())
        return out.permute(0, 2, 3, 1)


class IGAB(nn.Module):
    def __init__(self, dim, dim_head=64, heads=16, num_blocks=2):
        super().__init__()
        self.blocks = nn.ModuleList([])
        for _ in range(num_blocks):
            self.blocks.append(nn.ModuleList([
                IG_MSA(dim=dim, dim_head=dim_head, heads=heads),
                PreNorm(dim, FeedForward(dim=dim))
            ]))

    def forward(self, x, illu_fea):
        x = x.permute(0, 2, 3, 1)
        for (attn, ff) in self.blocks:
            x = attn(x, illu_fea_trans=illu_fea.permute(0, 2, 3, 1)) + x
            x = ff(x) + x
        out = x.permute(0, 3, 1, 2)
        return out


class Denoiser(nn.Module):
    def __init__(self, in_dim=3, out_dim=3, dim=31, level=2, num_blocks=[1, 1, 1]):
        super(Denoiser, self).__init__()
        self.dim = dim
        self.level = level

        self.embedding = nn.Conv2d(in_dim, self.dim, 3, 1, 1, bias=False)

        # Encoder
        self.encoder_layers = nn.ModuleList([])
        dim_level = dim
        for i in range(level):
            self.encoder_layers.append(nn.ModuleList([
                IGAB(dim=dim_level, num_blocks=num_blocks[i], dim_head=dim, heads=dim_level // dim),
                nn.Conv2d(dim_level, dim_level * 2, 4, 2, 1, bias=False),
                nn.Conv2d(dim_level, dim_level * 2, 4, 2, 1, bias=False)
            ]))
            dim_level *= 2

        # Bottleneck
        self.bottleneck = IGAB(dim=dim_level, dim_head=dim, heads=dim_level // dim, num_blocks=num_blocks[-1])

        # Decoder
        self.decoder_layers = nn.ModuleList([])
        for i in range(level):
            self.decoder_layers.append(nn.ModuleList([
                nn.ConvTranspose2d(dim_level, dim_level // 2, stride=2, kernel_size=2, padding=0),
                nn.Conv2d(dim_level, dim_level // 2, 1, 1, bias=False),
                IGAB(dim=dim_level // 2, num_blocks=num_blocks[level - 1 - i], dim_head=dim, heads=(dim_level // 2) // dim),
            ]))
            dim_level //= 2

        # Output projection
        self.mapping = nn.Conv2d(self.dim, out_dim, 3, 1, 1, bias=False)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            torch.nn.init.trunc_normal_(m.weight, std=.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward(self, x, illu_fea):
        fea = self.embedding(x)

        # Encoder Loop
        fea_encoder = []
        illu_fea_list = []
        for (igab_block, FeaDownSample, IlluFeaDownsample) in self.encoder_layers:
            fea = igab_block(fea, illu_fea)
            illu_fea_list.append(illu_fea)
            fea_encoder.append(fea)
            fea = FeaDownSample(fea)
            illu_fea = IlluFeaDownsample(illu_fea)

        # Bottleneck
        fea = self.bottleneck(fea, illu_fea)

        # Decoder Loop
        for i, (FeaUpSample, Fusion, LeWinBlock) in enumerate(self.decoder_layers):
            fea = FeaUpSample(fea)
            fea = Fusion(torch.cat([fea, fea_encoder[self.level - 1 - i]], dim=1))
            illu_fea = illu_fea_list[self.level - 1 - i]
            fea = LeWinBlock(fea, illu_fea)

        return self.mapping(fea) + x




class RetinexFormer(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, n_feat=32, level=2, num_blocks=[1, 1, 1]):
        super(RetinexFormer, self).__init__()
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.estimator = IlluminationEstimator(n_feat)
        self.denoiser = Denoiser(in_dim=in_channels, out_dim=out_channels, dim=n_feat, level=level, num_blocks=num_blocks)

    def forward(self, img):
        illu_fea, illu_map = self.estimator(img)
        input_img = img * illu_map
        output_img = self.denoiser(input_img, illu_fea)
        return output_img

    def fit(self, loader, total_iterations, criterion, optimizer, scheduler, save_path, model_name, itrs_k=1000):
        print(f"Training started...\n")
        criterion = criterion.to(self.device)
        self.to(self.device)
        self.train()
        itrs = 1
        tloss = 0
        st = time.time()
        while True:
            for x, y in loader:
                optimizer.zero_grad()
                x, y = x.to(self.device), y.to(self.device)
                enh = self(x)
                enh = torch.clamp(enh, 0, 1)
                loss = criterion(enh, y)
                tloss += loss.item()
                loss.backward()
                nn.utils.clip_grad_norm_(self.parameters(), max_norm=5.0)
                optimizer.step()
                scheduler.step()
                
                if itrs%itrs_k == 0:
                    l = tloss / itrs
                    et = time.time() - st
                    print(f"Iterations: {itrs}/{total_iterations}, train_loss: {l:.6f}, time-taken: {et:.2f} sec")
                    self.save_weights(os.path.join(save_path, model_name))
                    st = time.time()
                
                if itrs>=total_iterations:
                    l = tloss / itrs
                    et = time.time() - st
                    print(f"Iterations {itrs}/{total_iterations}, train_loss: {l:.6f}, time-taken: {et:.2f} sec")
                    self.save_weights(os.path.join(save_path, model_name))
                    print("\nTraining Completed!\n")
                    return
                itrs += 1

    def predict(self, x):
        self.to(self.device)
        self.eval()
        with torch.no_grad():
            return self(x.to(self.device))

    def save_weights(self, path):
        torch.save(self.state_dict(), path)
        print(f"Model saved at {path}\n")

    def load_weights(self, path):
        self.load_state_dict(torch.load(path, map_location=self.device))
        print(f"Model loaded from {path}\n")

    def calculate_psnr(self, low, high):
        enh = self.predict(low)
        mse = torch.mean((enh - high) ** 2)
        if mse == 0:
            return 100
        pixel_max = 1.0
        return 20 * torch.log10(pixel_max / torch.sqrt(mse))


In [ ]:

def augmentations(low, high):
    # 1. Flip
    r = random.random()
    if r > 0.5:
        low, high = TF.hflip(low), TF.hflip(high)
    else:
        low, high = TF.vflip(low), TF.vflip(high)

    # 2. Rotation 

    angle = random.choice([0, 90, 180, 270])
    low, high = TF.rotate(low, angle), TF.rotate(high, angle)

    low = TF.to_tensor(low)
    high = TF.to_tensor(high)

    return low, high


In [ ]:


class CustomDataset(Dataset):
    def __init__(self, low_img, high_img, aug=None, num_patches=1, patch_size=128, is_test=True, sample_size=10):
        super(CustomDataset, self).__init__()
        if sample_size:
            self.low_img = sorted(low_img)[:sample_size]
            self.high_img = sorted(high_img)[:sample_size]
        else:
            self.low_img = sorted(low_img)
            self.high_img = sorted(high_img)
        self.patch_size=patch_size
        if num_patches <= 0:
            raise ValueError("Number of patches should be greater than 0")
        else:
            self.num_patches=num_patches
        self.aug = aug
        self.test = is_test
        print(f"Number of total images: {len(self.low_img)}")

    def __len__(self):
        return len(self.low_img) * self.num_patches

    def __getitem__(self, idx):
        img_idx = idx % len(self.low_img)
        low_img = cv2.imread(self.low_img[img_idx])
        low_img = cv2.cvtColor(low_img, cv2.COLOR_BGR2RGB)

        W, H, _ = low_img.shape
        h = random.randint(0, H-self.patch_size)
        w = random.randint(0, W-self.patch_size)
        crop_box = (w, h, w+self.patch_size, h+self.patch_size)

        if self.test:
            low_img = low_img[w:w+self.patch_size, h:h+self.patch_size]
            return TF.to_tensor(low_img)

        high_img = cv2.imread(self.high_img[img_idx])
        high_img = cv2.cvtColor(high_img, cv2.COLOR_BGR2RGB)
        
        low_img = low_img[w:w+self.patch_size, h:h+self.patch_size]
        high_img = high_img[w:w+self.patch_size, h:h+self.patch_size]

        if self.aug:
            return self.aug(low_img, high_img)
        else:
            return TF.to_tensor(low_img), TF.to_tensor(high_img)


def get_loaders(low_img_paths, high_img_paths, img_ext, batch_size, augmentations, data_name, patch_size=128, num_patches=1):
    low_paths = glob.glob(os.path.join(low_img_paths, f'*.{img_ext}'))
    high_paths = glob.glob(os.path.join(high_img_paths, f'*.{img_ext}'))

    aug = None
    shuffle = False
    is_test = False

    if data_name.lower() == 'test':
        aug = augmentations
        shuffle = True
        is_test = True


    ds = CustomDataset(low_paths,
                        high_paths,
                        patch_size=patch_size,
                        num_patches=num_patches,
                        aug=aug,
                        sample_size=None,
                        is_test=is_test)

    loader = DataLoader(ds,
                        batch_size=batch_size,
                        num_workers=os.cpu_count(),
                        shuffle=shuffle,
                        pin_memory=True,
                        prefetch_factor=batch_size,
                        drop_last=True)

    return loader

In [ ]:
img_paths = "/kaggle/input/datasets/shaileshhkumarr/lol-dataset/LOLdataset"
batch_size=8
patch_size = 128
num_patches = 1


train_loader = get_loaders(low_img_paths = os.path.join(img_paths, 'train/low'),
                           high_img_paths = os.path.join(img_paths, 'train/high'),
                           batch_size=batch_size,
                           patch_size=patch_size,
                           num_patches=num_patches,
                           img_ext='png',
                           augmentations=augmentations,
                           data_name='train')

val_loader = get_loaders(low_img_paths=os.path.join(img_paths, 'val/low'),
                         high_img_paths=os.path.join(img_paths, 'val/high'),
                         batch_size=2,
                         patch_size=patch_size,
                         num_patches=num_patches,
                         img_ext='png',
                         data_name='val',
                         augmentations=None)


In [ ]:
save_path='./'
model_name='retinex-multihead-transformer.pth'
lr = 2e-4
lr_min = 1e-4
total_iterations = 50000

model = RetinexFormer()
criterion = nn.L1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.999))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_iterations, eta_min=lr_min)

In [ ]:
torch.backends.cudnn.benchmark=True

torch.set_float32_matmul_precision('high')

model = torch.compile(model)

In [ ]:
model.fit(loader=train_loader,
          total_iterations=total_iterations,
          itrs_k=2500,
          criterion=criterion,
          optimizer=optimizer,
          scheduler=scheduler,
          save_path=save_path,
          model_name=model_name)


In [ ]:
# model.load_weights("/kaggle/working/retinex-multihead-transformer.pth")
# model.to(device)

In [ ]:
psnr = 0
for x,y in tqdm(val_loader):
    x = x.to(device) 
    y = y.to(device)
    psnr += model.calculate_psnr(x, y)
psnr /= len(val_loader)       
print(f"PSNR on validation images: {psnr}")


In [ ]:
def enhance(loader, model=model, test=False, k=10):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    i = 0
    for x in tqdm(loader):
        if test:
            inp = x.to(device)
            with torch.no_grad():
                enh = model(inp)
                enh = torch.clamp(enh, 0, 1)
                for xi,ei in zip(inp, enh):
                    ei = ei.detach().cpu().permute(1, 2, 0).numpy()
                    # ei = cv2.cvtColor(ei, cv2.COLOR_BGR2RGB)
                    plt.subplot(1, 2, 1)
                    plt.imshow(xi.detach().cpu().permute(1, 2, 0).numpy())
                    plt.title('Low img')
                    plt.subplot(1, 2, 2)
                    plt.imshow(ei)
                    plt.title('Enhanced img')
                    plt.show()
            if i==k:
                break
            i += 1 
        else:
            inp = x[0].to(device)
            target = x[1].to(device)
            
            with torch.no_grad():
                enh = model(inp)
                enh = torch.clamp(enh, 0, 1)
                for xi,yi, ei in zip(inp, target, enh):
                    plt.subplot(1, 3, 1)
                    plt.imshow(xi.detach().cpu().permute(1, 2, 0).numpy())
                    plt.title('Low img')
                    plt.subplot(1, 3, 2)
                    plt.imshow(ei.detach().cpu().permute(1, 2, 0).numpy())
                    plt.title('Enhanced img')
                    plt.subplot(1, 3, 3)
                    plt.imshow(yi.detach().cpu().permute(1, 2, 0).numpy())
                    plt.title('Target img')
                    plt.show()
            if i==k:
                break
            i += 1


In [ ]:
enhance(val_loader, model, test=False)

In [ ]:
img_paths = "/kaggle/input/datasets/shaileshkumarvishwak/custom-llie-dataset/Custom-LLIE-Dataset"
batch_size=8
patch_size = 128
num_patches = 1


cust_train_loader = get_loaders(low_img_paths = os.path.join(img_paths, 'Train/low'),
                           high_img_paths = os.path.join(img_paths, 'Train/bright'),
                           batch_size=batch_size,
                           patch_size=patch_size,
                           num_patches=num_patches,
                           img_ext='jpg',
                           augmentations=augmentations,
                           data_name='train')

cust_val_loader = get_loaders(low_img_paths=os.path.join(img_paths, 'Val/low'),
                         high_img_paths=os.path.join(img_paths, 'Val/bright'),
                         batch_size=batch_size,
                         patch_size=256,
                         num_patches=num_patches,
                         img_ext='jpg',
                         data_name='val',
                         augmentations=None)

cust_test_loader = get_loaders(low_img_paths=os.path.join(img_paths, 'Test/low'),
                         high_img_paths=os.path.join(img_paths, 'Test/bright'),
                         batch_size=batch_size,
                         patch_size=256,
                         num_patches=num_patches,
                         img_ext='jpg',
                         data_name='test',
                         augmentations=None)


In [ ]:

save_path='./'
model_name='retinex-multihead-transformer.pth'
lr = 2e-4
lr_min = 1e-6
total_iterations = 100000

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_iterations, eta_min=lr_min)


In [ ]:
model.fit(loader=cust_train_loader,
          total_iterations=total_iterations,
          itrs_k=5000,
          criterion=criterion,
          optimizer=optimizer,
          scheduler=scheduler,
          save_path=save_path,
          model_name=model_name)

In [ ]:
psnr = 0
for x,y in tqdm(cust_val_loader):
    x = x.to(device) 
    y = y.to(device)
    psnr += model.calculate_psnr(x, y)
psnr /= len(cust_val_loader)       
print(f"PSNR on custom dataset-2 validation images: {psnr}") 

In [ ]:
enhance(cust_val_loader, model, test=False, k=4)

In [ ]:
enhance(cust_test_loader, model, test=True, k=4)

In [ ]:
import numpy as np
import time


class Enhancer:
    def __init__(self, model, batch_size):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model = model.to(self.device)
        self.batch_size = batch_size
    
    def get_ultra_sharp_mask(self, patch_size, fade_width=64): # Increased fade_width
        """
        Creates a very smooth 2D Cosine window. 
        Higher fade_width (64) ensures no sharp edges.
        """
        mask1d = torch.ones(patch_size)
        # Smoother Cosine taper
        ramp = 0.5 * (1 - torch.cos(torch.linspace(0, np.pi, fade_width)))
        mask1d[:fade_width] = ramp
        mask1d[-fade_width:] = torch.flip(ramp, [0])
        
        mask2d = mask1d.view(1, -1) * mask1d.view(-1, 1)
        return mask2d.unsqueeze(0) 
    
    def combine_tensor_patches(self, patch_tensors, coords, original_size, padded_size, patch_size):
        h, w = original_size
        nh, nw = padded_size
        canvas = torch.zeros((3, nh, nw), dtype=torch.float32)
        weight_sum = torch.zeros((1, nh, nw), dtype=torch.float32)
        
        # 64 pixel ka smooth transition boundary
        mask = self.get_ultra_sharp_mask(patch_size, fade_width=64)
        
        for idx, (i, j) in enumerate(coords):
            patch = patch_tensors[idx].float()
            # Ensure tensor is [0, 1]
            if patch.max() > 1.0: patch = patch / 255.0
                
            canvas[:, i:i+patch_size, j:j+patch_size] += (patch * mask)
            weight_sum[:, i:i+patch_size, j:j+patch_size] += mask
    
        # Division by weight_sum ensures patches blend perfectly
        full_tensor = canvas / (weight_sum + 1e-8)
        full_tensor = torch.clamp(full_tensor, 0, 1)
        
        img_np = full_tensor.permute(1, 2, 0).numpy()
        final_img = (img_np * 255.0).astype(np.uint8)
        
        return final_img[:h, :w, :]

    def enhance_image(self, img):
        # Image pre-processing
        if len(img.shape) == 3 and img.shape[2] == 3:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        else:
            img_rgb = img

        patch_size = 512
        stride = 256 # 50% overlap is crucial
        h, w, _ = img_rgb.shape
        
        # Better padding logic
        pad_h = (patch_size - h % stride) % stride + (patch_size - stride)
        pad_w = (patch_size - w % stride) % stride + (patch_size - stride)
        img_padded = cv2.copyMakeBorder(img_rgb, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT)
        nh, nw, _ = img_padded.shape
        
        patches = []
        coords = []
        for i in range(0, nh - patch_size + 1, stride):
            for j in range(0, nw - patch_size + 1, stride):
                p = img_padded[i:i+patch_size, j:j+patch_size, :]
                patches.append(torch.from_numpy(p).permute(2, 0, 1).float() / 255.0)
                coords.append((i, j))

        # Batch processing
        loader = DataLoader(patches, batch_size=self.batch_size)
        enhanced_list = []
        self.model.eval()
        st = time.time()
        with torch.no_grad():
            for batch in loader:
                out = self.model(batch.to(self.device))
                out = torch.clamp(out, 0, 1)
                enhanced_list.extend([p.cpu() for p in out])
                
        output = self.combine_tensor_patches(enhanced_list, coords, (h, w), (nh, nw), patch_size)
        return output, time.time() - st

In [ ]:
def load_img(path):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

In [ ]:
def enhance_whole_image(paths, model=model, test=False, k=10):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    enhancer_obj = Enhancer(model=model, batch_size=8)
    i = 0
    for path in paths:
        img = load_img(path)
        # inp = img.to(device)
        with torch.no_grad():
            enh, t = enhancer_obj.enhance_image(img)
            enh = cv2.cvtColor(enh, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(20,20))
            plt.subplot(1, 2, 1)
            plt.imshow(img)
            plt.title('Low img')
            plt.subplot(1, 2, 2)
            plt.imshow(enh)
            plt.title(f'Enhanced img {t} seconds')
            plt.show()
        if i==k:
            break
        i += 1 

In [ ]:
darkface_path=glob.glob("/kaggle/input/datasets/soumikrakshit/dark-face-dataset/image/*.png")

enhance_whole_image(darkface_path, model, test=True, k=50)